# Tugas Praktikum Big Data Pertemuan 10 — Dashboard Streaming Kopi Nusantara
**Nama:** Sholahuddin Ahmad  
**NPM:** 250506070

### Bagian A: Menyiapkan Streaming Source & Folder Simulasi

In [1]:
from spark_init import spark, F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
import os
import shutil
import time
import pandas as pd
from datetime import datetime

# Hentikan query aktif sebelumnya jika ada
for q in spark.streams.active:
    q.stop()

spark.conf.set("spark.sql.shuffle.partitions", "4")

folder_tugas = "tugas10_stream"
folder_output = "tugas10_output"
folder_checkpoint = "tugas10_checkpoint"

shutil.rmtree(folder_tugas, ignore_errors=True)
shutil.rmtree(folder_output, ignore_errors=True)
shutil.rmtree(folder_checkpoint, ignore_errors=True)

os.makedirs(folder_tugas, exist_ok=True)

skema_pesanan = StructType([
    StructField("pesanan_id", StringType(), True),
    StructField("nama_cabang", StringType(), True),
    StructField("nama_menu", StringType(), True),
    StructField("qty", IntegerType(), True),
    StructField("harga_satuan", IntegerType(), True),
    StructField("waktu_pesan", TimestampType(), True),
])

df_stream_tugas = (
    spark.readStream
    .format("csv")
    .schema(skema_pesanan)
    .option("header", True)
    .load(folder_tugas)
)

print("Streaming Source Bagian A Siap. isStreaming:", df_stream_tugas.isStreaming)


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/11/08 08:16:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


⚡ SparkSession siap! (Spark 3.5.9 | local[*])


Streaming Source Bagian A Siap. isStreaming: True


### Bagian B: Streaming Query Agregasi Omset per Cabang

In [2]:
for q in spark.streams.active:
    if q.name == "agregasi_cabang":
        q.stop()

df_dengan_total = df_stream_tugas.withColumn("total_harga", F.col("qty") * F.col("harga_satuan"))

agregasi_cabang = df_dengan_total.groupBy("nama_cabang").agg(
    F.sum("total_harga").alias("total_omset"),
    F.count("pesanan_id").alias("jumlah_pesanan")
)

query_cabang = agregasi_cabang.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_cabang") \
    .start()

print("Query Bagian B (agregasi_cabang) aktif:", query_cabang.isActive)

Query Bagian B (agregasi_cabang) aktif: True


### Bagian C: Streaming Query Window Waktu 2 Menit

In [3]:
for q in spark.streams.active:
    if q.name == "agregasi_waktu":
        q.stop()

df_stream_waktu = spark.readStream \
    .format("csv") \
    .schema(skema_pesanan) \
    .option("header", True) \
    .load(folder_tugas)

df_waktu_total = df_stream_waktu.withColumn("total_harga", F.col("qty") * F.col("harga_satuan"))

agregasi_waktu = df_waktu_total.groupBy(
    F.window(F.col("waktu_pesan"), "2 minutes")
).agg(
    F.sum("total_harga").alias("total_omset_window"),
    F.count("pesanan_id").alias("jumlah_pesanan")
)

query_waktu = agregasi_waktu.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_waktu") \
    .start()

print("Query Bagian C (agregasi_waktu) aktif:", query_waktu.isActive)

Query Bagian C (agregasi_waktu) aktif: True


### Bagian D: Simulasi Aliran 3 Batch Data Pesanan

In [4]:
# Batch 1
now = datetime.now()
pd.DataFrame({
    "pesanan_id": ["P001", "P002", "P003"],
    "nama_cabang": ["Magelang", "Yogyakarta", "Semarang"],
    "nama_menu": ["Kopi Susu Gula Aren", "Americano Dingin", "Caramel Macchiato"],
    "qty": [2, 1, 3],
    "harga_satuan": [22000, 20000, 28000],
    "waktu_pesan": [now.strftime("%Y-%m-%d %H:%M:%S")] * 3
}).to_csv(f"{folder_tugas}/batch_pesanan_1.csv", index=False)

time.sleep(2)
query_cabang.processAllAvailable()
query_waktu.processAllAvailable()
print("=== HASIL MONITORING CABANG SETELAH BATCH 1 ===")
spark.sql("SELECT * FROM agregasi_cabang ORDER BY total_omset DESC").show()

# Batch 2
now = datetime.now()
pd.DataFrame({
    "pesanan_id": ["P004", "P005", "P006"],
    "nama_cabang": ["Magelang", "Semarang", "Magelang"],
    "nama_menu": ["V60 Arabika Menoreh", "Tubruk Robusta Temanggung", "Cireng Bumbu Rujak"],
    "qty": [2, 4, 2],
    "harga_satuan": [26000, 16000, 15000],
    "waktu_pesan": [now.strftime("%Y-%m-%d %H:%M:%S")] * 3
}).to_csv(f"{folder_tugas}/batch_pesanan_2.csv", index=False)

time.sleep(2)
query_cabang.processAllAvailable()
query_waktu.processAllAvailable()
print("=== HASIL MONITORING CABANG SETELAH BATCH 2 ===")
spark.sql("SELECT * FROM agregasi_cabang ORDER BY total_omset DESC").show()

# Batch 3
now = datetime.now()
pd.DataFrame({
    "pesanan_id": ["P007", "P008", "P009", "P010"],
    "nama_cabang": ["Yogyakarta", "Yogyakarta", "Semarang", "Magelang"],
    "nama_menu": ["Espresso Nusantara", "Roti Bakar Cokelat", "Nasi Goreng Mercon", "Kopi Susu Gula Aren"],
    "qty": [3, 2, 1, 5],
    "harga_satuan": [18000, 18000, 28000, 22000],
    "waktu_pesan": [now.strftime("%Y-%m-%d %H:%M:%S")] * 4
}).to_csv(f"{folder_tugas}/batch_pesanan_3.csv", index=False)

time.sleep(2)
query_cabang.processAllAvailable()
query_waktu.processAllAvailable()
print("=== HASIL MONITORING CABANG SETELAH BATCH 3 ===")
spark.sql("SELECT * FROM agregasi_cabang ORDER BY total_omset DESC").show()

print("=== HASIL MONITORING WINDOW WAKTU 2 MENIT ===")
spark.sql("SELECT window.start, window.end, total_omset_window, jumlah_pesanan FROM agregasi_waktu").show(truncate=False)


=== HASIL MONITORING CABANG SETELAH BATCH 1 ===


+-----------+-----------+--------------+
|nama_cabang|total_omset|jumlah_pesanan|
+-----------+-----------+--------------+
|   Semarang|      84000|             1|
|   Magelang|      44000|             1|
| Yogyakarta|      20000|             1|
+-----------+-----------+--------------+



=== HASIL MONITORING CABANG SETELAH BATCH 2 ===


+-----------+-----------+--------------+
|nama_cabang|total_omset|jumlah_pesanan|
+-----------+-----------+--------------+
|   Semarang|     148000|             2|
|   Magelang|     126000|             3|
| Yogyakarta|      20000|             1|
+-----------+-----------+--------------+



=== HASIL MONITORING CABANG SETELAH BATCH 3 ===


+-----------+-----------+--------------+
|nama_cabang|total_omset|jumlah_pesanan|
+-----------+-----------+--------------+
|   Magelang|     236000|             4|
|   Semarang|     176000|             3|
| Yogyakarta|     110000|             3|
+-----------+-----------+--------------+

=== HASIL MONITORING WINDOW WAKTU 2 MENIT ===


+-------------------+-------------------+------------------+--------------+
|start              |end                |total_omset_window|jumlah_pesanan|
+-------------------+-------------------+------------------+--------------+
|2026-11-08 08:16:00|2026-11-08 08:18:00|522000            |10            |
+-------------------+-------------------+------------------+--------------+



### Bagian E: File Sink Parquet Berkelanjutan (Append Mode)

In [5]:
try:
    if "query_file_sink" in locals() and query_file_sink.isActive:
        query_file_sink.stop()
except Exception:
    pass

query_file_sink = df_stream_tugas.writeStream \
    .format("parquet") \
    .option("path", folder_output) \
    .option("checkpointLocation", folder_checkpoint) \
    .outputMode("append") \
    .start()

print("File sink Parquet aktif:", query_file_sink.isActive)

now = datetime.now()
pd.DataFrame({
    "pesanan_id": ["P011", "P012"],
    "nama_cabang": ["Solo", "Magelang"],
    "nama_menu": ["Caramel Macchiato", "Kentang Goreng Bolognese"],
    "qty": [2, 1],
    "harga_satuan": [28000, 22000],
    "waktu_pesan": [now.strftime("%Y-%m-%d %H:%M:%S")] * 2
}).to_csv(f"{folder_tugas}/batch_pesanan_4.csv", index=False)

time.sleep(4)
query_file_sink.processAllAvailable()

print("Daftar berkas fisik Parquet di folder output:")
for f in os.listdir(folder_output):
    if f.endswith(".parquet"):
        print("-", f)

df_verifikasi_parquet = spark.read.parquet(folder_output)
print(f"Total baris data tersimpan di Parquet: {df_verifikasi_parquet.count()} baris")
df_verifikasi_parquet.show()

File sink Parquet aktif: True


Daftar berkas fisik Parquet di folder output:
- part-00001-131c1eb6-5a2a-47e8-aa47-37d942d212d3-c000.snappy.parquet
- part-00000-436673fa-f9ce-48d7-a850-101e238ca1d9-c000.snappy.parquet


Total baris data tersimpan di Parquet: 12 baris


+----------+-----------+--------------------+---+------------+-------------------+
|pesanan_id|nama_cabang|           nama_menu|qty|harga_satuan|        waktu_pesan|
+----------+-----------+--------------------+---+------------+-------------------+
|      P007| Yogyakarta|  Espresso Nusantara|  3|       18000|2026-11-08 08:17:23|
|      P008| Yogyakarta|  Roti Bakar Cokelat|  2|       18000|2026-11-08 08:17:23|
|      P009|   Semarang|  Nasi Goreng Mercon|  1|       28000|2026-11-08 08:17:23|
|      P010|   Magelang| Kopi Susu Gula Aren|  5|       22000|2026-11-08 08:17:23|
|      P004|   Magelang| V60 Arabika Menoreh|  2|       26000|2026-11-08 08:17:19|
|      P005|   Semarang|Tubruk Robusta Te...|  4|       16000|2026-11-08 08:17:19|
|      P006|   Magelang|  Cireng Bumbu Rujak|  2|       15000|2026-11-08 08:17:19|
|      P001|   Magelang| Kopi Susu Gula Aren|  2|       22000|2026-11-08 08:17:03|
|      P002| Yogyakarta|    Americano Dingin|  1|       20000|2026-11-08 08:17:03|
|   

### Bagian F: Cleanup & Penutupan Seluruh Streaming Query

In [6]:
query_cabang.stop()
query_waktu.stop()
query_file_sink.stop()

print("Semua streaming query berhasil dihentikan dengan aman.")

Semua streaming query berhasil dihentikan dengan aman.
